# RAG Systems

Retrieval Augmented Generation

![rag](./images/rag1.png)

In [13]:
import tiktoken
from langchain_openai import OpenAIEmbeddings
import numpy as np
import bs4
from langchain_community.document_loaders import WebBaseLoader

import os
from dotenv import load_dotenv

load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'

langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = langchain_api_key

open_api_key = os.getenv("OPEN_API_KEY")

os.environ['OPENAI_API_KEY'] = open_api_key
os.environ['USER_AGENT'] = os.getenv("USER_AGENT")
USER_AGENT = os.getenv("USER_AGENT")

Let's try querying LLM without context

In [14]:
import openai

client = openai.OpenAI(api_key=open_api_key)


def get_completion(prompt, model="gpt-3.5-turbo"):
    response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Adjust creativity
        )
    return response.choices[0].message.content

prompt = "Хто такий Ігор Михайлович Черевко?"
response = get_completion(prompt)
print(response)

Ігор Михайлович Черевко - український політик, громадський діяч, народний депутат України. Він є членом фракції "Слуга народу" в Верховній Раді України. Ігор Михайлович Черевко відомий своєю активною участю в політичному житті України, а також своєю підтримкою реформ в країні.


## What is RAG system?

A RAG (Retrieval-Augmented Generation) system is an AI approach that combines information retrieval with generative AI to improve response accuracy and relevance. It works by first retrieving relevant documents or knowledge from a database and then using a generative model (like GPT) to process and summarize the information.

In [15]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [16]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

8

In [21]:
from langchain_openai import OpenAIEmbeddings

embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
random_result = embd.embed_query("Lorem ipsum dolor sit amet, consectetur adipiscing elit.")

print(len(document_result))
print(document_result[:20])

1536
[-0.014474782161414623, 0.005460602231323719, -0.021073397248983383, -0.02753557451069355, -0.014065469615161419, 0.021482709795236588, -0.01289954874664545, -0.020961767062544823, 0.0010845233919098973, -0.007045138161629438, 0.005314861889928579, 0.034258224070072174, 0.0007697867695242167, -0.01249643787741661, -0.017290355637669563, 0.01416469644755125, 0.03430783748626709, -0.006815674714744091, 0.02066408470273018, -0.023566482588648796]


In [ ]:
print(query_result)

In [23]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
# similarity = cosine_similarity(query_result, random_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.8806915835035412


In [37]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://mathmod.chnu.edu.ua/", "https://mathmod.chnu.edu.ua/pro-nas/spivrobitnyky/cherevko-ihor-mykhailovych/")
    # bs_kwargs=dict(
    #     parse_only=bs4.SoupStrainer(
    #         class_=("post-content", "post-title", "post-header")
    #     )
    # ),
)
blog_docs = loader.load()

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [ ]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)
print(splits)

# splits = blog_docs.split(' ')
# print(splits)

pip install langchain-community faiss-cpu


In [39]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings())

vectorstore.save_local("../models/")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [41]:
docs = retriever.invoke("Хто такий Ігор Михайлович Черевко?")
print(docs)

[Document(id='87f1dfd5-cb6d-474b-8b45-3ecd1f8f2bc3', metadata={'source': 'https://mathmod.chnu.edu.ua/pro-nas/spivrobitnyky/cherevko-ihor-mykhailovych/', 'title': 'Черевко Ігор Михайлович - Кафедра математичного моделювання', 'description': 'сторінка Черевка Ігоря Михайловича', 'language': 'uk'}, page_content='Ігор Михайлович Черевко розпочав трудову діяльність після закінчення у 1978 році Чернівецького державного університету на посаді асистента, з 1986 року працював старшим викладачем, доцентом кафедри прикладної математики, завідувачем кафедри математичного моделювання.'), Document(id='973885fc-08d2-4932-8991-3f3f1150d0e1', metadata={'source': 'https://mathmod.chnu.edu.ua/pro-nas/spivrobitnyky/cherevko-ihor-mykhailovych/', 'title': 'Черевко Ігор Михайлович - Кафедра математичного моделювання', 'description': 'сторінка Черевка Ігоря Михайловича', 'language': 'uk'}, page_content='i.cherevko@chnu.edu.ua\n\n\n\n\n\n\nЧеревко Ігор Михайлович\nДоктор фізико-математичних наук, професор\nЗа

In [42]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# also look up -> rag fusion, rag iterative prompting etc.

prompt = ChatPromptTemplate.from_template(template)

In [43]:
# LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [44]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("Хто такий Ігор Михайлович Черевко?")

'Ігор Михайлович Черевко - провідний спеціаліст в Україні з диференціально-функціональних рівнянь, доктор фізико-математичних наук, професор, завідувач кафедри математичного моделювання.'

![rag](./images/rag2.png)